# IMPORTACIÓN DE DATOS Y LIBRERÍAS

In [102]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

In [129]:
df = pd.read_csv(os.path.join('Datos', 'Transformados', 'df_meteorologico.csv'), index_col = 0)

In [133]:
nuevo = []
for i in range(df.shape[0]):
    if df['recurrence'].iloc[i] == 1:
        nuevo.append(True)
    elif df['recurrence'].iloc[i] > 1:
        nuevo.append(False)
    else:
        nuevo.append('REVISAR')

In [136]:
df['cliente_nuevo'] = nuevo

In [ ]:
datos = df
var_meteo = ['MEAN_tmin', 'MEAN_tmax', 'MEAN_tmed', 'MEAN_sol', 'MEAN_velmedia', 'MEAN_racha', 'MEAN_hrMedia']
for var in var_meteo:
    df = df[~df[str(var)].isna()]

# LIMPIEZA RÁPIDA

In [104]:
df['completed_entry_forms_count'] = df['completed_entry_forms_count'].fillna(value = 0)

# SELECCIÓN DE VARIABLES PARA LA CLUSTERIZACIÓN

In [105]:
X = df

In [106]:
cols_rm = ['F_C_I', 'F_C_O', 'IDEMA', 'idema_code', 'region', 'city',
   'MIN_tmin', 'Q1_tmin', 'Q2_tmin', 'Q3_tmin', 'MAX_tmin', 'IQR_tmin', 'STD_tmin',
   'MIN_tmax', 'Q1_tmax', 'Q2_tmax', 'Q3_tmax', 'MAX_tmax', 'IQR_tmax', 'STD_tmax',
   'MIN_tmed', 'Q1_tmed', 'Q2_tmed', 'Q3_tmed', 'MAX_tmed', 'IQR_tmed', 'STD_tmed',
   'MIN_prec', 'Q1_prec', 'Q2_prec', 'Q3_prec', 'MAX_prec', 'IQR_prec', 'STD_prec',
   'MIN_sol', 'Q1_sol', 'Q2_sol', 'Q3_sol', 'MAX_sol', 'IQR_sol', 'STD_sol',
   'MIN_velmedia', 'Q1_velmedia', 'Q2_velmedia', 'Q3_velmedia', 'MAX_velmedia', 'IQR_velmedia', 'STD_velmedia',
   'MIN_racha', 'Q1_racha', 'Q2_racha', 'Q3_racha', 'MAX_racha', 'IQR_racha', 'STD_racha',
   'MIN_hrMedia', 'Q1_hrMedia', 'Q2_hrMedia', 'Q3_hrMedia', 'MAX_hrMedia', 'IQR_hrMedia', 'STD_hrMedia']
for var in cols_rm:
    del X[str(var)]

In [107]:
vars_fecha = ['booked_at', 'checkin_time', 'checkout_time', 'asset_opening_date', 'last_entry_form_completed_at', 'cancelled_at']
for var in vars_fecha:
    del X[str(var)]

In [108]:
vars_superfluas = ['brand', 'available_units', 'bought_products', 'rate', 'cancellation_lead_time', 'status', 'stay_length', 'requested_category_name', 'travel_agency_name']
for var in vars_superfluas:
    del X[str(var)]

In [109]:
var_bool = ['all_entry_forms_completed', 'returning_inhabitant', 'libere_community']
mapping = {'yes': 1, 'no': 0}
for var in var_bool:
    X[str(var)] = X[str(var)].map(mapping)

In [110]:
var_boolean = var_bool + ['is_cancelled']
for var in var_boolean:
    X[str(var)] = X[str(var)].astype('bool')

In [111]:
del var_bool
del var_boolean
del cols_rm
del mapping
del var
del vars_fecha
del vars_superfluas

# FORMATEO DE LAS VARIABLES SELECCIONADAS PARA LA CLUSTERIZACION

In [112]:
X['checkin_month'] = X['checkin_month'].map({1: 'ENE', 2: 'FEB', 3: 'MAR', 
                        4: 'ABR', 5: 'MAY', 6: 'JUN', 
                        7: 'JUL', 8: 'AGO', 9: 'SEP', 
                        10: 'OCT', 11: 'NOV', 12: 'DIC'})
X['checkin_day'] = X['checkin_day'].map({1: 'LUN', 2: 'MAR', 3: 'MIE', 4: 'JUE', 5: 'VIE', 6: 'SAB', 7: 'DOM'})

In [113]:
var_num = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'int64']['index'].to_list() + X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'float64']['index'].to_list()
var_cat = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'object']['index'].to_list()
var_bool = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'bool']['index'].to_list()

In [114]:
medias = []
desves = []
for var in var_num:
    medias.append(float(X[str(var)].mean()))
    desves.append(float(X[str(var)].std()))
normalizadores = pd.DataFrame({'MU': medias, 'SIGMA': desves, 'VAR': var_num})

In [115]:
for var in var_num:
    normalizer = StandardScaler()
    X[str(var)] = normalizer.fit_transform(X[[str(var)]])

In [116]:
X = pd.get_dummies(X, columns = var_cat)

In [117]:
for i in X.columns.to_list():
    if X[i].isna().sum() != 0:
        print(i)

days_before_cancel


In [118]:
del X['days_before_cancel']

In [124]:
X = X[X['is_cancelled']]
del X['is_cancelled']

OUTLIERS

In [ ]:
var_num = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'int64']['index'].to_list() + X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'float64']['index'].to_list()
outliers_bool = pd.DataFrame()
for var in var_num:
    outliers_bool[str(var)] = (X[str(var)] > 2.7)
outliers_na = X[var_num][~outliers_bool]
index_outliers = []
for i in range(outliers_na.shape[0]):
    if outliers_na.iloc[i].isna().sum() != 0:
        index_outliers.append(i)
X = X.drop(X.index[index_outliers]).reset_index()

In [ ]:
del X['index']

CLUSTERIZACIÓN

1. KMEANS

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score, make_scorer
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV

In [ ]:
kmeans = KMeans(random_state = 4) #MODELO BASE, CON LOS PARAMETROS FIJOS

#PARÁMETROS PARA BUSCAR EL MEJOR MODELO
param_grid_kmeans = {
    'n_clusters': [3,4,5], #Número de clústers, literalmente el K de Kmeans
    'init': ['k-means++', 'random'], #Método para inicializar los centroides
    'n_init': [10,20] #Número de veces que el algoritmo se ejecuta con diferentes inicializaciones. Luego, se guarda el resultado con menos inercia (el mejor).
}

#SELECCIÓN DE LA MÉTRICA DE ÉXITO
scorer = make_scorer(silhouette_score)

#CREACIÓN GRIDSEARCH
grid = GridSearchCV(
    estimator = kmeans,
    param_grid = param_grid_kmeans,
    scoring = scorer,
    cv = 2, #NO TIENE SENTIDO HACER CROSS VALIDATION, PERO GRIDSEARCH LO EXIGE
    n_jobs = -1
)

In [ ]:
# HAY SOSPECHA DE QUE GRID SEARCH NO FUNCIONE CORRECTAMENTE CON KMEANS

# import warnings
# warnings.filterwarnings('ignore')

# #AJUSTE DE GRIDSEARCH A LOS DATOS
# grid.fit(X)

# #MEJORES PARÁMETROS E INDICE DE SILUETA
# print("Mejores parámetros:", grid.best_params_)
# print("Mejor score:", grid.best_score_)

# #MEJOR MODELO Y RESULTADOS
# best_kmeans = grid.best_estimator_
# labels = best_kmeans.labels_
# best_kmeans_params = best_kmeans.get_params()
# kmeans = KMeans(algorithm = best_kmeans_params['algorithm'],
#                 copy_x = best_kmeans_params['copy_x'],
#                 init = best_kmeans_params['init'], 
#                 max_iter = best_kmeans_params['max_iter'],
#                 n_clusters = best_kmeans_params['n_clusters'],
#                 n_init = best_kmeans_params['n_init'],
#                 random_state = best_kmeans_params['random_state'],
#                 tol = best_kmeans_params['tol'],
#                 verbose = best_kmeans_params['verbose'])
# clusters_kmeans = kmeans.fit_predict(X)
# silhouette_avg_kmeans = silhouette_score(X, clusters_kmeans)
# centroides_kmeans = kmeans.cluster_centers_
# print('RESULTADO DE KMEANS:')
# print(f'Para {best_kmeans_params['n_clusters']} clusters, el índice de silueta es: {silhouette_avg_kmeans}')

HAY QUE QUITARLES EL COMENTADO A LAS DOS CELDAS DE ABAJO

In [ ]:
# kmeans_silhouette = {}
# for k in param_grid_kmeans['n_clusters']:
#     for init in param_grid_kmeans['init']:
#         for n_init in param_grid_kmeans['n_init']:
#             kmeans = KMeans(n_clusters = k, 
#                             random_state = 4, 
#                             init = init,
#                             n_init = n_init)
#             clusters_kmeans = kmeans.fit_predict(X)
#             key = f'{k}_{init}_{n_init}'
#             kmeans_silhouette[key] = silhouette_score(X, clusters_kmeans)
#             print(f'Kmeans {key} hecho.')
# dict_to_df = {
#     'MODELO': list(kmeans_silhouette.keys()),
#     'IND_SILUETA': list(kmeans_silhouette.values())
# }
# kmeans_silhouette = pd.DataFrame(dict_to_df)
# print(f'El modelo con mejor índice de silueta es: {kmeans_silhouette[kmeans_silhouette['IND_SILUETA'].max() == kmeans_silhouette['IND_SILUETA']]['MODELO'].to_list()[0]} y el índice de silueta es de {kmeans_silhouette[kmeans_silhouette['IND_SILUETA'].max() == kmeans_silhouette['IND_SILUETA']]['IND_SILUETA'].to_list()[0]}')

CLUSTERIZACIÓN JERÁRQUICA

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram

In [ ]:
def plot_dendrogram(model, **kwargs):
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count
    linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    dendrogram(linkage_matrix, **kwargs)

In [ ]:
hclust = AgglomerativeClustering()

param_grid_hclust_1 = {
    'n_clusters': [3, 4, 5],
    'metric': ['euclidean', 'l1', 'l2', 'manhattan'],
    'linkage': ['complete', 'average', 'single']
}

param_grid_hclust_2 = {
    'n_clusters': [3, 4, 5],
    'metric': ['euclidean'],
    'linkage': ['ward']
}

scorer = make_scorer(silhouette_score)

grid_hclust_1 = GridSearchCV(
    estimator = hclust,
    param_grid = param_grid_hclust_1,
    scoring = scorer,
    cv = 2
)

grid_hclust_2 = GridSearchCV(
    estimator = hclust,
    param_grid = param_grid_hclust_2,
    scoring = scorer,
    cv = 2
)

In [ ]:
#TARDA MUCHO EN EJECUTARSE, POR ELLO SE DESCARTA

# grid_hclust_1.fit(X)
# print("Mejores parámetros:", grid_hclust_1.best_params_)
# print("Mejor score:", grid_hclust_1.best_score_)
# grid_hclust_2.fit(X)
# print("Mejores parámetros:", grid_hclust_2.best_params_)
# print("Mejor score:", grid_hclust_2.best_score_)

DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
eps_values = [1.5, 1.75, 2]
min_samples_values = [45, 50]
metric_options = ['manhattan', 'euclidean']

best_score = -1
best_params = {}

for dist in metric_options:
    for eps in eps_values:
        for min_samples in min_samples_values:
            dbscan = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric=dist
            )
            
            labels = dbscan.fit_predict(X)

            mask = labels != -1
            labels_no_noise = labels[mask]
            X_no_noise = X[mask]

            if len(set(labels_no_noise)) < 2:
                continue

            score = silhouette_score(X_no_noise, labels_no_noise)

            if score > best_score:
                best_score = score
                best_params = {
                    'eps': eps,
                    'min_samples': min_samples,
                    'metric': dist
                }

best_dbscan = DBSCAN(**best_params)
labels = best_dbscan.fit_predict(X)

clusters1 = pd.DataFrame({'CLUSTERS':(labels.tolist())})

print("Mejores parámetros:", best_params)
print("Mejor Silhouette Score:", best_score)
print(f'Distribución de {clusters1.value_counts().to_list()}')

Mejores parámetros: {'eps': 1.75, 'min_samples': 50, 'metric': 'euclidean'}
Mejor Silhouette Score: 0.1081544437709432
Distribución de [38881, 1250, 484, 150, 110, 94, 85, 57, 51, 50, 50]


In [ ]:
X = X.iloc[clusters1[clusters1['CLUSTERS'] == -1].index]

In [ ]:
eps_values = [1.5, 1.75, 2]
min_samples_values = [50, 55]
metric_options = ['euclidean', 'manhattan']

best_score = -1
best_params = {}

for dist in metric_options:
    for eps in eps_values:
        for min_samples in min_samples_values:
            dbscan = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric=dist
            )
            
            labels = dbscan.fit_predict(X)

            mask = labels != -1
            labels_no_noise = labels[mask]
            X_no_noise = X[mask]

            if len(set(labels_no_noise)) < 2:
                continue

            score = silhouette_score(X_no_noise, labels_no_noise)

            if score > best_score:
                best_score = score
                best_params = {
                    'eps': eps,
                    'min_samples': min_samples,
                    'metric': dist
                }

best_dbscan = DBSCAN(**best_params)
labels = best_dbscan.fit_predict(X)

clusters1 = pd.DataFrame({'CLUSTERS':(labels.tolist())})

print("Mejores parámetros:", best_params)
print("Mejor Silhouette Score:", best_score)
print(f'Distribución de {clusters1.value_counts().to_list()}')

Mejores parámetros: {'eps': 2, 'min_samples': 55, 'metric': 'euclidean'}
Mejor Silhouette Score: 0.27355054373840765
Distribución de [37660, 435, 225, 191, 97, 80, 76, 60, 57]


In [ ]:
X = X.iloc[clusters1[clusters1['CLUSTERS'] == -1].index]

In [ ]:
eps_values = [2, 2.25, 2.33]
min_samples_values = [55, 60]
metric_options = ['euclidean', 'manhattan']

best_score = -1
best_params = {}

for dist in metric_options:
    for eps in eps_values:
        for min_samples in min_samples_values:
            dbscan = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric=dist
            )
            
            labels = dbscan.fit_predict(X)

            mask = labels != -1
            labels_no_noise = labels[mask]
            X_no_noise = X[mask]

            if len(set(labels_no_noise)) < 2:
                continue

            score = silhouette_score(X_no_noise, labels_no_noise)

            if score > best_score:
                best_score = score
                best_params = {
                    'eps': eps,
                    'min_samples': min_samples,
                    'metric': dist
                }

best_dbscan = DBSCAN(**best_params)
labels = best_dbscan.fit_predict(X)

clusters1 = pd.DataFrame({'CLUSTERS':(labels.tolist())})

print("Mejores parámetros:", best_params)
print("Mejor Silhouette Score:", best_score)
print(f'Distribución de {clusters1.value_counts().to_list()}')

Mejores parámetros: {'eps': 2.25, 'min_samples': 60, 'metric': 'euclidean'}
Mejor Silhouette Score: 0.16400609831513902


In [ ]:
X = X.iloc[clusters1[clusters1['CLUSTERS'] == -1].index]

In [ ]:
eps_values = [1.75, 1.80]
min_samples_values = [45, 49]
metric_options = ['euclidean', 'manhattan']

best_score = -1
best_params = {}

for dist in metric_options:
    for eps in eps_values:
        for min_samples in min_samples_values:
            dbscan = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric=dist
            )
            
            labels = dbscan.fit_predict(X)

            mask = labels != -1
            labels_no_noise = labels[mask]
            X_no_noise = X[mask]

            if len(set(labels_no_noise)) < 2:
                continue

            score = silhouette_score(X_no_noise, labels_no_noise)

            if score > best_score:
                best_score = score
                best_params = {
                    'eps': eps,
                    'min_samples': min_samples,
                    'metric': dist
                }

best_dbscan = DBSCAN(**best_params)
labels = best_dbscan.fit_predict(X)

clusters1 = pd.DataFrame({'CLUSTERS':(labels.tolist())})

print("Mejores parámetros:", best_params)
print("Mejor Silhouette Score:", best_score)
print(f'Distribución de {clusters1.value_counts().to_list()}')

Mejores parámetros: {}
Mejor Silhouette Score: -1
Distribución de [34941, 12, 11, 11, 9, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]


In [ ]:
# n_clusters = 3
# clusterer = KMeans(n_clusters=n_clusters, random_state=10) #MODELO
# cluster_labels = clusterer.fit_predict(X) #LISTA DE LA VARIABLE CON LOS CLUSTERES
# silhouette_avg = silhouette_score(X, cluster_labels) #ÍNDICE MEDIO DE SILUETA
# print("For n_clusters =", n_clusters, ", The average silhouette_score is :", silhouette_avg)
# sample_silhouette_values = silhouette_samples(X, cluster_labels) #ÍNDICE DE SILUETA DE LOS VALORES DE LA VARIABLE
# centers = clusterer.cluster_centers_ #CENTROIDES